In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Load dataset
transform = transforms.ToTensor()
train_dataset = torchvision.datasets.MNIST(
    root='../data/',
    train=True,
    download=True,
    transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Noise function
def add_gaussian_noise(image_tensor, noise_factor=0.3):
    noise = torch.randn_like(image_tensor)
    noisy_image = image_tensor + noise_factor * noise
    return torch.clamp(noisy_image, 0., 1.)

print("Setup complete!")

Setup complete!


In [2]:
class DenoisingVAE(nn.Module):
    def __init__(self):
        super(DenoisingVAE, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.fc_mu = nn.Linear(64 * 7 * 7, 20)
        self.fc_logvar = nn.Linear(64 * 7 * 7, 20)
        self.fc_decode = nn.Linear(20, 64 * 7 * 7)
        self.dec_conv1 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.dec_conv2 = nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(-1, 64 * 7 * 7)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        x = F.relu(self.fc_decode(z))
        x = x.view(-1, 64, 7, 7)
        x = F.relu(self.dec_conv1(x))
        return torch.sigmoid(self.dec_conv2(x)), mu, logvar


class ExpertCNN(nn.Module):
    def __init__(self):
        super(ExpertCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def vae_loss_function(reconstructed_x, original_x, mu, logvar):
    BCE = F.binary_cross_entropy(reconstructed_x, original_x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

classification_loss = nn.CrossEntropyLoss()

print("Blueprints and loss functions ready!")

Blueprints and loss functions ready!


In [3]:
healer = DenoisingVAE()
expert = ExpertCNN()

# Load your already-trained weights
healer.load_state_dict(torch.load('../models/healer_vae.pth', weights_only=True))
expert.load_state_dict(torch.load('../models/expert_cnn.pth', weights_only=True))

# Set to TRAINING mode so both can learn together
healer.train()
expert.train()

# Combined optimizer - updates BOTH brains at once
# Small lr=1e-4 so we don't destroy what they already know
combined_params = list(healer.parameters()) + list(expert.parameters())
combined_optimizer = torch.optim.Adam(combined_params, lr=1e-4)

print("Pre-trained brains loaded!")
print("Ready for End-to-End Fine-Tuning!")

Pre-trained brains loaded!
Ready for End-to-End Fine-Tuning!


In [4]:
#end to end training
epochs = 3

print("Starting End-to-End Training...")
print("-" * 45)

for epoch in range(epochs):
    total_combined_loss = 0

    for batch_idx, (clean_images, labels) in enumerate(train_loader):

        # 1. Add noise
        noisy_images = add_gaussian_noise(clean_images, noise_factor=0.5)

        # 2. Reset gradients for BOTH models
        combined_optimizer.zero_grad()

        # 3. Full pipeline: Noisy -> Healer -> Expert -> Prediction
        healed_images, mu, logvar = healer(noisy_images)
        predictions = expert(healed_images)

        # 4. Combined loss
        loss_vae = vae_loss_function(healed_images, clean_images, mu, logvar)
        loss_cnn = classification_loss(predictions, labels)
        total_loss = loss_vae + (10.0 * loss_cnn)  # CNN weighted higher

        # 5. Update BOTH brains in one step
        total_loss.backward()
        combined_optimizer.step()

        total_combined_loss += total_loss.item()

    avg_loss = total_combined_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/3 complete  |  Combined Loss: {avg_loss:.4f}")

print("-" * 45)
print("Fine-Tuning Complete!")

Starting End-to-End Training...
---------------------------------------------
Epoch 1/3 complete  |  Combined Loss: 118.6484
Epoch 2/3 complete  |  Combined Loss: 118.3424
Epoch 3/3 complete  |  Combined Loss: 118.0554
---------------------------------------------
Fine-Tuning Complete!


In [5]:
#Save the Fine-Tuned Models
import os
os.makedirs('../models', exist_ok=True)

# Save as NEW files so you don't overwrite your originals
torch.save(healer.state_dict(), '../models/healer_vae_finetuned.pth')
torch.save(expert.state_dict(), '../models/expert_cnn_finetuned.pth')

print("Fine-tuned models saved!")
print("  ../models/healer_vae_finetuned.pth")
print("  ../models/expert_cnn_finetuned.pth")

Fine-tuned models saved!
  ../models/healer_vae_finetuned.pth
  ../models/expert_cnn_finetuned.pth


In [6]:
from torchvision import datasets

# Load test data
test_dataset = datasets.MNIST(root='../data/', train=False, download=True, transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Switch to evaluation mode
healer.eval()
expert.eval()

correct_noisy  = 0
correct_healed = 0
total          = 0

with torch.no_grad():
    for clean_images, labels in test_loader:
        noisy_images          = add_gaussian_noise(clean_images, noise_factor=0.5)
        healed_images, _, _   = healer(noisy_images)

        pred_noisy  = expert(noisy_images).argmax(dim=1)
        pred_healed = expert(healed_images).argmax(dim=1)

        total          += labels.size(0)
        correct_noisy  += (pred_noisy  == labels).sum().item()
        correct_healed += (pred_healed == labels).sum().item()

acc_noisy  = 100 * correct_noisy  / total
acc_healed = 100 * correct_healed / total

print("===== FINE-TUNED RESULTS =====")
print(f"Noisy (no healing):          {acc_noisy:.2f}%")
print(f"Fine-Tuned Self-Healing:     {acc_healed:.2f}%")
print(f"Accuracy recovered:          +{acc_healed - acc_noisy:.2f}%")
print("==============================")

===== FINE-TUNED RESULTS =====
Noisy (no healing):          47.92%
Fine-Tuned Self-Healing:     93.82%
Accuracy recovered:          +45.90%
